# Build de APK de React Native / Expo no Google Colab

Este notebook configura o ambiente Android no Google Colab para compilar um aplicativo React Native (ou Expo com prebuild) e gerar um arquivo `.apk` de forma 100% automatizada.

## 1. Instalar o Android SDK e Java 17
Esta célula prepara todo o ecossistema Android necessário para rodar o build no servidor Linux do Colab.

In [ ]:
%%bash
# 1. Atualizar pacotes e instalar o Java 17, unzip, cmake e ninja-build
apt-get update -qq && apt-get install -y openjdk-17-jdk wget unzip cmake ninja-build -qq > /dev/null

# 2. Criar diretório para o SDK
export ANDROID_HOME=/opt/android-sdk
mkdir -p $ANDROID_HOME/cmdline-tools

# 3. Baixar o Android Command Line Tools
wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -O android_tools.zip

# 4. Descompactar e ajustar a estrutura de pastas
unzip -q android_tools.zip -d $ANDROID_HOME/cmdline-tools
mv $ANDROID_HOME/cmdline-tools/cmdline-tools $ANDROID_HOME/cmdline-tools/latest
rm android_tools.zip

# 5. Exportar variáveis de ambiente
export PATH=$PATH:$ANDROID_HOME/cmdline-tools/latest/bin:$ANDROID_HOME/platform-tools

# 6. Aceitar as licenças do SDK
yes | sdkmanager --licenses > /dev/null

# 7. Instalar as plataformas e build-tools
sdkmanager "platform-tools" "platforms;android-34" "build-tools;34.0.0" > /dev/null
echo "Android SDK configurado com sucesso!"

## 2. Clonar o Repositório e Instalar Pacotes
Preencha a URL do seu repositório no GitHub para cloná-lo e instalar as dependências NPM. Se for um app Expo Managed, esta célula gerará a pasta `/android` automaticamente.

In [ ]:
%%bash
# Substitua a URL abaixo pela URL do seu repositório do GitHub
GIT_REPO_URL="https://github.com/SEU_USUARIO/SEU_REPOSITORIO.git"
PROJECT_DIR="/content/meu_projeto"

rm -rf $PROJECT_DIR
git clone $GIT_REPO_URL $PROJECT_DIR

cd $PROJECT_DIR

# Instalar as dependências do Node.js
npm install

# IMPORTANTE: Se você estiver usando Expo Managed Workflow e o repositório não
# tiver a pasta 'android', geramos a pasta nativa com o prebuild:
npx expo prebuild --platform android

## 3. Fazer o Build do APK
Escolha uma das duas opções abaixo para rodar o build:

### Opção A: APK de Debug (Desenvolvimento)
Compilação mais rápida, assinada automaticamente com uma chave de debug autogerada.

In [ ]:
%%bash
export ANDROID_HOME=/opt/android-sdk
export PATH=$PATH:$ANDROID_HOME/cmdline-tools/latest/bin:$ANDROID_HOME/platform-tools
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64

PROJECT_DIR="/content/meu_projeto"
cd $PROJECT_DIR/android

chmod +x gradlew
./gradlew assembleDebug

### Opção B: APK de Release / Preview (Otimizado - RECOMENDADO)
Gera um APK de produção super leve e de alta performance. O script cria uma chave keystore temporária de testes diretamente no Colab e assina o app dinamicamente, permitindo instalar o aplicativo diretamente no celular, exatamente como o `eas build --profile preview`.

In [ ]:
%%bash
export ANDROID_HOME=/opt/android-sdk
export PATH=$PATH:$ANDROID_HOME/cmdline-tools/latest/bin:$ANDROID_HOME/platform-tools
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64

PROJECT_DIR="/content/meu_projeto"
cd $PROJECT_DIR/android

# 1. Gerar chave temporária no Colab
keytool -genkey -v -keystore temp.keystore -alias temp-key-alias -keyalg RSA -keysize 2048 -validity 10000 -storepass 123456 -keypass 123456 -dname "CN=ColabTest, OU=Development, O=Test, L=Colab, S=SP, C=BR" -noprompt

# 2. Permissão e Compilação em Release injetando a chave gerada
chmod +x gradlew
./gradlew assembleRelease \
  -Pandroid.injected.signing.store.file=$PROJECT_DIR/android/temp.keystore \
  -Pandroid.injected.signing.store.password=123456 \
  -Pandroid.injected.signing.key.alias=temp-key-alias \
  -Pandroid.injected.signing.key.password=123456

## 4. Baixar o APK para a sua Máquina
Esta célula em Python baixa o arquivo gerado (Release ou Debug) diretamente para a sua máquina local.

In [ ]:
from google.colab import files
import os

apk_debug = "/content/meu_projeto/android/app/build/outputs/apk/debug/app-debug.apk"
apk_release = "/content/meu_projeto/android/app/build/outputs/apk/release/app-release.apk"

if os.path.exists(apk_release):
    print("Baixando o APK de Release otimizado...")
    files.download(apk_release)
elif os.path.exists(apk_debug):
    print("Baixando o APK de Debug...")
    files.download(apk_debug)
else:
    print("APK não encontrado. Verifique se o build ocorreu com sucesso nos logs anteriores.")